In [ ]:
!apt clean
!apt update && apt upgrade -y
!apt-get install -y zstd
!apt-get install lshw
!pip install pyngrok
!pip install -q timm albumentations transformers
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import torch
print("GPU Active:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name())
print("CUDA Cores :", torch.cuda.device_count())

In [ ]:
import os
import cv2
import gc
import time
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DistilBertModel

# Globale Konfiguration für Performance
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = os.cpu_count()
SEED = 42

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print(f"Nutze Gerät: {DEVICE} mit {NUM_WORKERS} CPU-Kernen.")

In [ ]:
import os
import cv2
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Hardware optimal auslasten
NUM_WORKERS = os.cpu_count()  # Nutzt alle verfügbaren CPU-Kerne für das Laden

# 2. Blitzschnelle Bild-Transformationen (Albumentations ist ca. 2x schneller als Torchvision PIL)
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),  # Konvertiert direkt in PyTorch Tensors
])

# 3. Den in Rust geschriebenen FAST Tokenizer laden
TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)

class KaggleMultimodalDataset(Dataset):
    def __init__(self, image_paths, texts, labels, transforms):
        self.image_paths = image_paths
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # BILD-LADEN: cv2.imread ist massiv schneller als PIL.Image.open!
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            image = self.transforms(image=image)["image"]
            
        # TEXT-LADEN: KEIN Padding hier! Wir übergeben nur den rohen Text.
        # Das Tokenisieren passiert parallel und dynamisch im Smart Collator.
        text = self.texts[idx]
        label = self.labels[idx]
        
        return {"image": image, "text": text, "label": label}

# 4. Der "Smart Collator" für Dynamic Padding
def multimodal_smart_collator(batch):
    """
    Diese Funktion sammelt ein Batch auf den CPU-Workers ein, 
    tokenisiert alle Texte gleichzeitig und füllt sie NUR bis zur 
    maximalen Länge DIESES Batches auf (spart bis zu 50% Rechenzeit).
    """
    images = torch.stack([item["image"] for item in batch])
    labels = torch.stack([item["label"] for item in batch])
    texts = [item["text"] for item in batch]
    
    # Batch-Tokenisierung nutzt Multithreading der HuggingFace Rust-Engine
    tokenized_text = tokenizer(
        texts,
        padding=True,          # Dynamic Padding: füllt nur bis zur längsten Sequenz im aktuellen Batch auf
        truncation=True,       # Schneidet ab, falls ein Text das Limit (z.B. 512) überschreitet
        max_length=128,        # Maximal erlaubte Obergrenze
        return_tensors="pt"    # Direkt als PyTorch Tensors ausgeben
    )
    
    return {
        "images": images,
        "input_ids": tokenized_text["input_ids"],
        "attention_mask": tokenized_text["attention_mask"],
        "labels": labels
    }

# --- BEISPIEL FÜR DIE INITIALISIERUNG ---
# Erstelle Dummy-Daten (Ersetze dies mit deinen echten Pfaden und Texten)
dummy_img_paths = ["sample_img.jpg"] * 1000  # Stelle sicher, dass die Bilder existieren
dummy_texts = ["Dies ist ein sehr kurzer Beispieltext für Kaggle."] * 1000
dummy_labels = np.random.rand(1000, 1)

# Dataset & DataLoader instanziieren
train_dataset = KaggleMultimodalDataset(dummy_img_paths, dummy_texts, dummy_labels, train_transforms)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,                      # T4-GPUs lieben Zweierpotenzen (32, 64, 128)
    shuffle=True,
    num_workers=NUM_WORKERS,            # Lädt Bilder parallel
    collate_fn=multimodal_smart_collator, # Aktiviert die ultraschnelle Textverarbeitung
    pin_memory=True,                    # Ermöglicht asynchronen GPU-Transfer
    persistent_workers=True,            # Verhindert Lags zwischen den Epochen
    drop_last=True                      # Wichtig für reibungsloses Multi-GPU Training
)

In [ ]:
import torch
import torch.nn as nn
import timm
from transformers import DistilBertModel

class KagglePerformanceMultimodalModel(nn.Module):
    def __init__(self, text_model_name="distilbert-base-uncased", num_classes=1, dropout_prob=0.3):
        super().__init__()
        
        # 1. TEXT BRANCH: DistilBERT (Schnell und speicherschonend)
        # Wir laden nur das Core-Modell ohne den Classification-Head
        self.text_encoder = DistilBertModel.from_pretrained(text_model_name)
        text_features_dim = self.text_encoder.config.hidden_size # Standard: 768
        
        # 2. VISION BRANCH: Vision Transformer (ViT) oder effizientes CNN (z.B. ResNet / ConvNeXt)
        # 'vit_tiny_patch16_224' oder 'resnet34' eignen sich hervorragend für schnelle Durchläufe
        self.vision_encoder = timm.create_model(
            'vit_tiny_patch16_224', 
            pretrained=True, 
            num_classes=0 # 0 entfernt den originalen Classification-Head -> Feature Extraction
        )
        vision_features_dim = self.vision_encoder.num_features # Gibt die Feature-Dimension zurück
        
        # 3. FUSION LAYER (Late Fusion)
        # Wir hängen beide Feature-Vektoren nebeneinander an (Concatenation)
        combined_dim = text_features_dim + vision_features_dim
        
        # 4. CLASSIFICATION HEAD (MLP)
        self.fusion_classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.BatchNorm1d(512), # Stabilisiert das Training bei FP16 massiv
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(512, num_classes) # num_classes=1 für Regression/Binäre Klassifikation
        )

    def forward(self, images, input_ids, attention_mask):
        # --- TEXT FEATURES EXTRACTION ---
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Wir nutzen den CLS-Token (Index 0) als Repräsentation des gesamten Textes
        text_features = text_outputs.last_hidden_state[:, 0, :] 
        
        # --- VISION FEATURES EXTRACTION ---
        vision_features = self.vision_encoder(images)
        
        # --- MULTIMODALE FUSION ---
        # Verbinde Text (Batch, 768) und Bild (Batch, Dim) zu (Batch, 768 + Dim)
        fused_features = torch.cat((text_features, vision_features), dim=1)
        
        # --- KLASSIFIKATION ---
        output = self.fusion_classifier(fused_features)
        return output

In [ ]:
import subprocess, time, os
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_MAX_QUEUE"] = "512"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_SCHED_SPREAD"] = "true"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_NOPRUNE"] = "true"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q4_0"
os.environ["OLLAMA_NO_CLOUD"] = "true"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["OLLAMA_GPU_OVERHEAD"] = "1000"
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull ministral-3:8b

In [ ]:
import subprocess

# 1. Definiere die Parameter im Modelfile-Format
modelfile_content = """
FROM ministral-3:8b
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
"""

# 2. Schreibe das Modelfile temporär auf die Festplatte
with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# 3. Erstelle das angepasste Modell (wir nennen es 'my-ministral')
subprocess.run(["ollama", "create", "my-ministral", "-f", "./Modelfile"])
print("🎉 Angepasstes Modell mit Temperatur-Settings wurde erstellt!")

In [ ]:
import torch
import numpy as np

print("🎉 Starte Multimodalen Sanity Check...")

# 1. Pipeline-Komponenten überprüfen
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-> Nutze Gerät: {device} (Verfügbare GPUs: {torch.cuda.device_count()})")

try:
    # 2. Modell initialisieren, kompilieren und auf GPU schieben
    print("-> Initialisiere Modell...")
    model = KagglePerformanceMultimodalModel().to(device)
    
    # Für den schnellen Sanity Check überspringen wir torch.compile(), 
    # testen aber direkt das Multi-GPU Setup (DataParallel)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
        
    model.train()

    # 3. Künstliche Rohdaten für 4 Samples erstellen (Minimale Batch-Größe)
    print("-> Generiere künstliche Testdaten...")
    mock_images = [np.uint8(np.random.rand(300, 300, 3) * 255) for _ in range(4)]
    mock_texts = [
        "Das ist der erste extrem kurze Testtext.",
        "Hier folgt ein etwas längerer Text für den Smart Collator Test.",
        "Kurz.",
        "Und der vierte und letzte Text in diesem simulierten Batch-Durchlauf."
    ]
    mock_labels = np.random.rand(4, 1)

    # 4. Dataset & DataLoader füttern
    test_dataset = KaggleMultimodalDataset(mock_images, mock_texts, mock_labels, train_transforms)
    
    # Wichtig: Für den Test im Dataset cv2.imread umgehen, da wir Arrays direkt übergeben
    test_dataset.image_paths = mock_images 
    def mock_getitem(self, idx):
        # Override für den Test, um keine echten Bilddateien von der Festplatte zu laden
        image = self.image_paths[idx]
        if self.transforms:
            image = self.transforms(image=image)["image"]
        return {"image": image, "text": self.texts[idx], "label": self.labels[idx]}
    KaggleMultimodalDataset.__getitem__ = mock_getitem

    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=4, collate_fn=multimodal_smart_collator
    )

    # 5. Ein einzelnes Batch abgreifen
    print("-> Teste DataLoader und Dynamic Padding...")
    batch = next(iter(test_loader))

    # Daten asynchron auf die GPU schieben
    images = batch["images"].to(device, non_blocking=True)
    input_ids = batch["input_ids"].to(device, non_blocking=True)
    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)

    print(f"   [INFO] Tensor-Shape Bilder: {images.shape}")
    print(f"   [INFO] Tensor-Shape Text (Input IDs): {input_ids.shape} <--- (Dynamisch angepasst!)")

    # 6. Forward & Backward Pass mit Mixed Precision (FP16)
    print("-> Teste Forward & Backward Pass (FP16)...")
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    scaler = torch.cuda.amp.GradScaler()

    optimizer.zero_grad(set_to_none=True)
    
    with torch.cuda.amp.autocast():
        outputs = model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print(f"   [INFO] Berechneter Loss: {loss.item():.4f}")
    print("\n✅ Check bestanden (siehe Log).")

except Exception as e:
    print("\n❌ FEHLER im Sanity Check entdeckt:")
    import traceback
    traceback.print_exc()

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("NGROK_KEY")

In [ ]:
import subprocess, time, requests
subprocess.Popen(["ngrok", "http", "11434", "--request-header-add", "ngrok-skip-browser-warning:true"])
time.sleep(3)
tunnels = requests.get("http://localhost:4040/api/tunnels").json
print = (tunnels["tunnels"][0]["public_url"])